# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/24f2001824/ml-flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1: Longer content is associated with higher visibility

The paper reports that longer content groups show higher average impressions and query coverage.

My methodology question is: how were the content-length groups defined, and how different are the groups in other factors such as page age, content type, or existing visibility? Since this is an observational comparison, I would treat the result as an observed association rather than evidence that increasing word count directly causes higher visibility.

### Finding 2: Refreshed pages show stronger performance

The paper reports that refreshed mature pages have higher observed impressions compared with stale pages.

My methodology question is: how was a page classified as refreshed, and were the refreshed and stale groups comparable before the refresh? In particular, I would want to check whether previous visibility, page age, or page selection could explain part of the difference. Without a time-aware or causal design, I would treat this as a directional association rather than proof that refreshing a page caused the increase.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In Week 5, I evaluated the model using a page-level split. Here I re-run the same model using a client-grouped split.

The grouped split keeps pages from the same client together, so the test set contains clients that were not used during training. I compare the two splits using the same model and Precision@50.

In [15]:
# This cell is for CODE (numbers, a query, a check).

!git clone https://github.com/24f2001824/ml-flyrank.git

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("/content/ml-flyrank/data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

features = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[features]
y = df["is_declining_label"]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent"
]

numeric_features = [
    col for col in features if col not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

def precision_at_50(y_true, scores):
    result = pd.DataFrame({
        "actual": y_true,
        "score": scores
    }).sort_values("score", ascending=False)

    return result.head(50)["actual"].mean()

fatal: destination path 'ml-flyrank' already exists and is not an empty directory.


### Before: Random page split

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

pipeline.fit(X_train, y_train)

random_scores = pipeline.predict_proba(X_test)[:, 1]

random_precision = precision_at_50(
    y_test.reset_index(drop=True),
    random_scores
)

print("Random page split Precision@50:", round(random_precision, 4))

Random page split Precision@50: 1.0


### After: Client-grouped split

In [17]:
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

pipeline.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_scores = pipeline.predict_proba(
    X_test_grouped
)[:, 1]

grouped_precision = precision_at_50(
    y_test_grouped.reset_index(drop=True),
    grouped_scores
)

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("Client-grouped split Precision@50:", round(grouped_precision, 4))
print("Clients in both train and test:", len(train_clients & test_clients))

Client-grouped split Precision@50: 1.0
Clients in both train and test: 0


### Before vs after

In [18]:
comparison = pd.DataFrame({
    "Split": [
        "Random page split",
        "Client-grouped split"
    ],
    "Precision@50": [
        random_precision,
        grouped_precision
    ]
})

comparison

,Split,Precision@50
0,Random page split,1.0
1,Client-grouped split,1.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I checked the final feature set for direct target leakage and identifiers that should not be used as predictive features.

The declining label is derived from trend_direction, so trend_direction and trend_pct are excluded from the model features. client_id and content_id are also excluded from the predictive feature set.

I also checked that the final feature list does not contain the target or other fields that directly encode the target.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_candidates = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id"
]

print("Leakage audit")

for col in leakage_candidates:
    if col in features:
        print(col, ": IN FEATURE SET")
    else:
        print(col, ": excluded")

print()
print("Number of model features:", len(features))

print()
print("Target distribution:")
print(df["is_declining_label"].value_counts(normalize=True))

Leakage audit
trend_direction : excluded
trend_pct : excluded
is_declining_label : excluded
client_id : excluded
content_id : excluded

Number of model features: 31

Target distribution:
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My original claim was:

"The Random Forest predicts which pages need to be refreshed."

After the validation and leakage audit, I would rewrite this as:

"The Random Forest produced a ranked list of pages associated with the observed declining-page label and achieved the measured Precision@50 on the held-out data."

This result is directional and should be used as decision-support for human review. It does not prove that refreshing a page will improve traffic or that the model identifies causal SEO factors.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.